# Lab 2: HFL

This lab presents the findings and uses part of the experimental methodology from the [original Federated Learning](https://arxiv.org/pdf/1602.05629.pdf) paper. In horizontal federated learning, all clients have access to the same complete model architecture, which they train on local data, sharing information about model updates but not their data.

Before starting, make sure to follow the overall setup for the labs.

<a href="https://blogs.nvidia.com/blog/what-is-federated-learning/" target="_blank"><img src="https://blogs.nvidia.com/wp-content/uploads/2019/10/federated_learning_animation_still_white.png" alt="FL Visualization" style="width:50%;"></a>

---

Before anything else, we download, load, and preprocess the [MNIST dataset](https://archive.ics.uci.edu/dataset/683/mnist+database+of+handwritten+digits), which we will use for the following experiments.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

data_path = "./data"
ETA = "\N{GREEK SMALL LETTER ETA}"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

torch.backends.cudnn.deterministic = True

transform = transforms.Compose([
    transforms.ToTensor(),
    # normalize by training set mean and standard deviation
    # resulting data has mean=0 and std=1
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(data_path, train=True, download=True, transform=transform)
test_loader = DataLoader(
    datasets.MNIST(data_path, train=False, download=False, transform=transform),
    # decrease batch size if running into memory issues when testing
    # a bespoke generator is passed to avoid reproducibility issues
    shuffle=False, drop_last=False, batch_size=10000, generator=torch.Generator())

We can then define a small convolutional neural network that will serve as our model.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
class MnistCnn(nn.Module):
    def __init__(self):
        super(MnistCnn, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)

        return output

With that, we can define a helper method, which, given a model, a loader for iterating through a set of data, and an optimizer for updating the model trains one epoch (i.e., learns going through all the available data once).

In [ ]:
from torch.optim import Optimizer


def train_epoch(model: nn.Module, loader: DataLoader, optimizer: Optimizer) -> None:
    model.train()

    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

We also define another utility method that splits the dataset into several chunks.

We assign samples within chunks in an IID (independent and identically distributed) fashion or allow only two labels to exist in each.

In [ ]:
from typing import cast

import numpy as np
import numpy.random as npr
from torch.utils.data import Subset


def split(nr_clients: int, iid: bool, seed: int) -> list[Subset]:
    rng = npr.default_rng(seed)

    if iid:
        splits = np.array_split(rng.permutation(len(train_dataset)), nr_clients)
    else:
        sorted_indices = np.argsort(np.array([target for _data, target in train_dataset]))
        shards = np.array_split(sorted_indices, 2 * nr_clients)
        shuffled_shard_indices = rng.permutation(len(shards))
        splits = [
            np.concatenate([shards[i] for i in shard_ind_pairs], dtype=np.int64)
            for shard_ind_pairs in shuffled_shard_indices.reshape(nr_clients, 2)]

    return [Subset(train_dataset, split) for split in cast(list[list[int]], splits)]

In [ ]:
sample_split = split(100, True, 42)

We define a short class for holding the results of training runs and the parameters used.

In [ ]:
from dataclasses import asdict, dataclass, field

from pandas import DataFrame


@dataclass
class RunResult:
    algorithm: str
    n: int      # number of clients
    c: float    # client_fraction
    b: int      # take -1 as inf
    e: int      # nr_local_epochs
    lr: float   # printed as lowercase eta
    seed: int
    wall_time: list[float] = field(default_factory=list)
    message_count: list[int] = field(default_factory=list)
    test_accuracy: list[float] = field(default_factory=list)

    def as_df(self, skip_wtime=True) -> DataFrame:
        self_dict = {
            k.capitalize().replace("_", " "): v
            for k, v in asdict(self).items()}

        if self_dict["B"] == -1:
            self_dict["B"] = "\N{INFINITY}"

        df = DataFrame({"Round": range(1, len(self.wall_time) + 1), **self_dict})
        df = df.rename(columns={"Lr": ETA})
        if skip_wtime:
            df = df.drop(columns=["Wall time"])
        return df

We create an abstract class as a template for all distributed learning clients, defining a method for outputting an update after training a given model on local data.

In [ ]:
from abc import ABC, abstractmethod


class Client(ABC):
    def __init__(self, client_data: Subset, batch_size: int) -> None:
        self.model = MnistCnn().to(device)
        self.loader_train = DataLoader(
            client_data, batch_size=batch_size, shuffle=False, drop_last=False)

    @abstractmethod
    def update(self, in_params: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        ...

On the flip side, a server needs to be able to run the (distributed) training process for a given number of rounds and test the current model it possesses.

In [ ]:
class Server(ABC):
    def __init__(self, lr: float, batch_size: int, seed: int) -> None:
        self.clients: list[Client]
        self.lr = lr
        self.batch_size = batch_size
        self.seed = seed
        torch.manual_seed(seed)
        self.model = MnistCnn().to(device)

    @abstractmethod
    def run(self, nr_rounds: int) -> RunResult:
        ...

    def test(self) -> float:
        # TODO
        ...

Over the previously defined server template, we can even formulate a centralized variant, which does not involve clients, as a precursor to distributed versions.

In [ ]:
from time import perf_counter

from torch.optim import SGD
from tqdm import trange


class CentralizedServer(Server):
    def __init__(self, lr: float, batch_size: int, seed: int) -> None:
        super().__init__(lr, batch_size, seed)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.loader_train = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
        self.clients = []

    def run(self, nr_rounds: int) -> RunResult:
        # TODO
        ...

In [ ]:
centralized_server = CentralizedServer(0.5, 1024, 42)
result_centralized = centralized_server.run(5)
centralized_df = result_centralized.as_df()
centralized_df

We can extend the template with some setup steps common to all decentralized algorithms.

In [ ]:
class DecentralizedServer(Server):
    def __init__(
            self, lr: float, batch_size: int, client_subsets: list[Subset],
            client_fraction: float, seed: int) -> None:
        # TODO
        ...

The two federated learning algorithms from the paper follow, alongside an overview of metric plotting.

---

For the FedSGD algorithm, the baseline from the paper, we first need to define the client, and we choose to pass gradients from the client as the update result.

In [ ]:
class GradientClient(Client):
    def __init__(self, client_data: Subset) -> None:
        super().__init__(client_data, len(client_data))

    def update(self, in_params: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        # TODO
        ...

We then define the corresponding server.

In [ ]:
class FedSgdServer(DecentralizedServer):
    def __init__(
            self, lr: float,
            client_subsets: list[Subset], client_fraction: float, seed: int) -> None:
        super().__init__(lr, -1, client_subsets, client_fraction, seed)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.clients = [GradientClient(subset) for subset in client_subsets]

    def run(self, nr_rounds: int) -> RunResult:
        # TODO
        ...

In [ ]:
fedsgd_gradient_server = FedSgdServer(0.02, sample_split, 0.2, 42)
result_fedsgd_gradient = fedsgd_gradient_server.run(5)
fedsgd_gradient_df = result_fedsgd_gradient.as_df()
fedsgd_gradient_df

The FedAvg algorithm is the paper's main contribution, requiring a client that passes around weights instead of gradients.

In [ ]:
class WeightClient(Client):
    def __init__(self, client_data: Subset, lr: float, batch_size: int, nr_epochs: int) -> None:
        super().__init__(client_data, batch_size)
        self.optimizer = SGD(params=self.model.parameters(), lr=lr)
        self.nr_epochs = nr_epochs

    def update(self, in_params: list[torch.Tensor], seed: int) -> list[torch.Tensor]:
        # TODO
        ...

Following that, we define the actual server code for the method.

In [ ]:
class FedAvgServer(DecentralizedServer):
    def __init__(
            self, lr: float, batch_size: int, client_subsets: list[Subset],
            client_fraction: float, nr_local_epochs: int, seed: int) -> None:
        super().__init__(lr, batch_size, client_subsets, client_fraction, seed)
        self.name = "FedAvg"
        self.nr_local_epochs = nr_local_epochs
        self.clients = [
            WeightClient(subset, lr, batch_size, nr_local_epochs)
            for subset in client_subsets]

    def run(self, nr_rounds: int) -> RunResult:
        # TODO
        ...

In [ ]:
fedavg_server = FedAvgServer(0.02, 200, sample_split, 0.2, 2, 42)
result_fedavg = fedavg_server.run(5)
fedavg_df = result_fedavg.as_df()
fedavg_df

Finally, we look at a quick example of plotting the accuracy per round of the two algorithms.

In [ ]:
import pandas as pd
import seaborn as sns

df = pd.concat([fedavg_df, fedsgd_gradient_df], ignore_index=True)
ax = sns.lineplot(df, x="Round", y="Test accuracy", hue="Algorithm", seed=0)
_ = ax.set_xticks(df["Round"].unique())

---

## Bonus: Memory-efficient memory representations

Below, we test the effect of representing a (small) language model using different numerical data types for the weights.

We use the Hugging Face Transformers framework to load and interface with the model variants.

In [ ]:
from time import perf_counter

import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, PreTrainedModel)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model_id = "HuggingFaceTB/SmolLM3-3B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenized_input = tokenizer(
    "The most important concept in distributed deep learning is", return_tensors="pt").to(device)
input_ids = tokenized_input["input_ids"]
attention_mask = tokenized_input["attention_mask"]


def benchmark_model(model: PreTrainedModel, dtype_name: str) -> None:
    print("Data type: ", dtype_name)

    memory_footprint_bytes = model.get_memory_footprint()
    print(
        f"Memory footprint (MiB): {(memory_footprint_bytes / (1024 ** 2)):.2f}")

    # Warm-up run
    _ = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=5,
        # Avoid warnings
        temperature=None,
        top_p=None,
    )

    start_time = perf_counter()
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        # Avoid warnings
        temperature=None,
        top_p=None,
    )
    elapsed_time = perf_counter() - start_time
    print(
        f"Speed (tokens/s): {((outputs.numel() - input_ids.numel()) / elapsed_time):.2f}")

    output_text = " ".join(tokenizer.decode(
        outputs[0], skip_special_tokens=True).split())
    print("Output:", output_text)

Let us load and test the model in standard single-precision (FP32) format.

In [ ]:
benchmark_model(
    AutoModelForCausalLM.from_pretrained(model_id, device_map=device, dtype=torch.float32),
    "FP32")

Up next, let us try a half-precision format (BF16 or FP16), which is often the default for trained language model weights (including the tested one):

In [ ]:
benchmark_model(
    AutoModelForCausalLM.from_pretrained(model_id, device_map=device, dtype=torch.bfloat16),
    "BF16")

Memory consumption is almost halved, speed is close to double, and output quality is maintained.

Finally, we can try a quantized version that uses integers instead of floats, further halving precision.
Since there is no native INT8 version of our model, we will use BitsAndBytes to quantize and dequantize the model on the fly.
Compared to a properly pre-quantized model, we will lose some memory savings and, more crucially, our generation speed will degrade somewhat compared to the single-precision case:

In [ ]:
benchmark_model(
    AutoModelForCausalLM.from_pretrained(
        model_id, device_map=device, quantization_config=BitsAndBytesConfig(load_in_8bit=True)),
    "INT8")

Note that, even with a pre-quantized model, to achieve further improvements in generation speed, we would also need hardware that supports lower-than-single-precision formats natively.